# 7.10 Convolutional Neural Networks

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook introduces convolutions as structure-aware layers for images and other grid-like data. The examples use small tensors so the mechanics stay visible.


## 7.10.1 Convolution intuition

### Why It Matters
A convolutional filter scans a small kernel across an image and produces a response map. The easiest way to see it is to apply a hand-crafted edge filter.


In [ ]:
import torch
import torch.nn.functional as F

image = torch.tensor([[[[0., 0., 0., 0.],
                        [0., 1., 1., 0.],
                        [0., 1., 1., 0.],
                        [0., 0., 0., 0.]]]])
kernel = torch.tensor([[[[-1., -1., -1.],
                         [-1.,  8., -1.],
                         [-1., -1., -1.]]]])
feature_map = F.conv2d(image, kernel, padding=1)
feature_map[0, 0].tolist()


### Interpretation
The kernel is highlighting changes in local intensity, which is the basic idea behind learned visual features.


## 7.10.2 Kernels, stride, and padding

### Why It Matters
Stride controls how far the filter jumps each step, and padding controls whether the border is preserved. Output shape changes immediately when you change them.


In [ ]:
import torch
from torch import nn

x = torch.randn(2, 1, 8, 8)
same = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
downsample = nn.Conv2d(1, 4, kernel_size=3, stride=2, padding=1)
{
    "same_shape": list(same(x).shape),
    "stride_2_shape": list(downsample(x).shape),
}


### Interpretation
These shape changes are the bookkeeping you need to understand before building deeper CNNs.


## 7.10.3 Pooling

### Why It Matters
Pooling summarizes nearby activations and reduces spatial resolution. Max pooling is easy to inspect because it keeps the strongest local response.


In [ ]:
import torch
from torch import nn

x = torch.tensor([[[[1., 2., 0., 1.], [3., 1., 4., 0.], [1., 5., 2., 2.], [0., 1., 3., 4.]]]])
pooled = nn.MaxPool2d(kernel_size=2, stride=2)(x)
pooled[0, 0].tolist()


### Interpretation
Pooling shrinks the map while preserving the strongest response in each local neighborhood.


## 7.10.4 Feature maps

### Why It Matters
A convolutional layer usually learns several filters, which means it outputs several feature maps rather than one.


In [ ]:
import torch
from torch import nn

x = torch.randn(3, 1, 8, 8)
conv = nn.Conv2d(1, 6, kernel_size=3, padding=1)
out = conv(x)
{"input_shape": list(x.shape), "feature_map_shape": list(out.shape), "num_learned_filters": conv.weight.shape[0]}


### Interpretation
Each output channel is the response of a different learned detector.


## 7.10.5 Classifier head

### Why It Matters
After convolutional feature extraction, the network usually flattens or pools features and passes them into a dense classification head.


In [ ]:
import torch
from torch import nn

model = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(4 * 4 * 4, 10),
)
x = torch.randn(5, 1, 8, 8)
out = model(x)
{"classifier_logits_shape": list(out.shape)}


### Interpretation
The head translates spatial features into task-specific predictions such as digit classes.


## 7.10.6 Build a small CNN for image classification

### Why It Matters
This final subsection trains a tiny CNN on the classic handwritten-digits dataset reshaped as images.


In [ ]:
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = torch.tensor(digits.images / 16.0, dtype=torch.float32).unsqueeze(1)
y = torch.tensor(digits.target, dtype=torch.long)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=33, stratify=y)

model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.Flatten(),
    nn.Linear(16 * 4 * 4, 10)
)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()
for _ in range(20):
    opt.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    opt.step()
with torch.no_grad():
    preds = model(X_test).argmax(dim=1)
    acc = (preds == y_test).float().mean().item()
{"test_accuracy": round(float(acc), 3), "final_loss": round(float(loss.item()), 4)}


### Interpretation
Even a small CNN can learn useful spatial features from very compact image inputs.
